# 00 - Bronze inventory of WennSoft tables
This inventories the tables in the bronze schema before modelling for silver and gold medallions. This creates two small delta tables: metadata.table_inventory and metadata.column_inventory.

In [1]:
from datetime import datetime, timezone
import re
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, LongType
)

BRONZE_SCHEMA = "bronze"
METADATA_SCHEMA = "metadata"
RUN_ROW_COUNTS = False       # Change to True only after the first successful run
SAMPLE_TABLE = ""           # Example: SV000810; leave blank to skip sampling
SAMPLE_ROWS = 20

# Prefixes are only discovery hints; classification does not change the data.
WENNSOFT_PREFIX_PATTERN = r"^(SV|WS|JC|JCM|EMS|EM|ME|K2A)"
GP_PREFIX_PATTERN = r"^(GL|RM|PM|SOP|POP|IV|UPR|FA|MC|SY|CM|BM|PA|TX)[A-Z_]*[0-9]"

observed_at = datetime.now(timezone.utc).isoformat()
print(f"Inventory timestamp (UTC): {observed_at}")

StatementMeta(, 257d2a4f-fa1d-48e1-b902-7869b767f2a7, 3, Finished, Available, Finished, True)

Inventory timestamp (UTC): 2026-09-10T14:55:51.949198+00:00


Discover tables and columns

In [2]:
def quote_identifier(value: str) -> str:
    return f"`{value.replace('`', '``')}`"

def classify_table(table_name: str) -> str:
    name = table_name.upper()
    if re.match(WENNSOFT_PREFIX_PATTERN, name):
        return "WennSoft candidate"
    if re.match(GP_PREFIX_PATTERN, name):
        return "Dynamics GP candidate"
    return "Other / custom"

try:
    show_tables_df = spark.sql(f"SHOW TABLES IN {quote_identifier(BRONZE_SCHEMA)}")
    table_rows = show_tables_df.collect()
except Exception as exc:
    raise RuntimeError(
        f"Could not read schema '{BRONZE_SCHEMA}'. Attach the correct Lakehouse, "
        "make it the default, and verify the schema name."
    ) from exc

table_names = sorted(row["tableName"] for row in table_rows)
if not table_names:
    raise RuntimeError(f"No tables were found in schema '{BRONZE_SCHEMA}'.")

print(f"Found {len(table_names):,} Bronze tables.")

StatementMeta(, 257d2a4f-fa1d-48e1-b902-7869b767f2a7, 4, Finished, Available, Finished, False)

Found 8 Bronze tables.


In [3]:
table_records = []
column_records = []

for table_name in table_names:
    qualified_name = (
        f"{quote_identifier(BRONZE_SCHEMA)}.{quote_identifier(table_name)}"
    )
    classification = classify_table(table_name)
    row_count = None
    inventory_error = None
    column_count = 0

    try:
        described = spark.sql(f"DESCRIBE TABLE {qualified_name}").collect()
        ordinal = 0
        for item in described:
            column_name = (item["col_name"] or "").strip()
            if not column_name or column_name.startswith("#"):
                continue
            ordinal += 1
            column_count += 1
            upper_column = column_name.upper()
            key_candidate = bool(re.search(
                r"(^ID$|_ID$|ID$|NMBR$|NUMBR$|NUMBER$|KEY$)", upper_column
            ))
            column_records.append({
                "source_schema": BRONZE_SCHEMA,
                "table_name": table_name,
                "ordinal_position": ordinal,
                "column_name": column_name,
                "data_type": item["data_type"],
                "comment": item["comment"],
                "key_name_candidate": key_candidate,
                "observed_at_utc": observed_at
            })
    except Exception as exc:
        inventory_error = f"DESCRIBE failed: {type(exc).__name__}: {str(exc)[:500]}"

    if RUN_ROW_COUNTS and inventory_error is None:
        try:
            row_count = spark.table(qualified_name).count()
        except Exception as exc:
            inventory_error = f"COUNT failed: {type(exc).__name__}: {str(exc)[:500]}"

    table_records.append({
        "source_schema": BRONZE_SCHEMA,
        "table_name": table_name,
        "classification": classification,
        "column_count": column_count,
        "row_count": row_count,
        "inventory_error": inventory_error,
        "observed_at_utc": observed_at
    })

StatementMeta(, 257d2a4f-fa1d-48e1-b902-7869b767f2a7, 5, Finished, Available, Finished, False)

Review the inventory of tables

In [4]:
table_schema = StructType([
    StructField("source_schema", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("classification", StringType(), False),
    StructField("column_count", LongType(), False),
    StructField("row_count", LongType(), True),
    StructField("inventory_error", StringType(), True),
    StructField("observed_at_utc", StringType(), False)
])

column_schema = StructType([
    StructField("source_schema", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("ordinal_position", LongType(), False),
    StructField("column_name", StringType(), False),
    StructField("data_type", StringType(), True),
    StructField("comment", StringType(), True),
    StructField("key_name_candidate", BooleanType(), False),
    StructField("observed_at_utc", StringType(), False)
])

table_inventory_df = spark.createDataFrame(table_records, schema=table_schema)
column_inventory_df = spark.createDataFrame(column_records, schema=column_schema)

display(table_inventory_df.orderBy("classification", "table_name"))

StatementMeta(, 257d2a4f-fa1d-48e1-b902-7869b767f2a7, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b637268d-3d15-4b11-8438-c44bb5441747)

In [5]:
display(
    table_inventory_df
        .groupBy("classification")
        .agg(
            F.count("*").alias("table_count"),
            F.sum("column_count").alias("column_count")
        )
        .orderBy("classification")
)

StatementMeta(, 257d2a4f-fa1d-48e1-b902-7869b767f2a7, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cae103c6-88bb-47cb-ba17-abde702fe33f)

In [6]:
display(
    column_inventory_df
        .filter(F.col("key_name_candidate"))
        .select("table_name", "ordinal_position", "column_name", "data_type")
        .orderBy("table_name", "ordinal_position")
)

StatementMeta(, 257d2a4f-fa1d-48e1-b902-7869b767f2a7, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 201bad51-06d9-489e-b0c8-cd174c88cdd2)

Save the metadata tables

In [7]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {quote_identifier(METADATA_SCHEMA)}")

table_inventory_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{METADATA_SCHEMA}.table_inventory")

column_inventory_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{METADATA_SCHEMA}.column_inventory")

print("Saved metadata.table_inventory and metadata.column_inventory")

StatementMeta(, 257d2a4f-fa1d-48e1-b902-7869b767f2a7, 9, Finished, Available, Finished, False)

Saved metadata.table_inventory and metadata.column_inventory
